<a href="https://colab.research.google.com/github/philippenchev98/rfm-customer-segmentation/blob/main/RFM_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#Данните са изтеглени директно от сървъра на UCI
print("Зареждане на данните... (може да отнеме около минута, тъй като файлът съдържа половин милион кортежа")
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"
df = pd.read_excel(url)

#Показваме първите 5 реда
print("ДАННИТЕ СА ЗАРЕДЕНИ УСПЕШНО!")
display(df.head())

print(f"\nОбщ размер на масива: {df.shape[0]} реда и {df.shape[1]} колони")

Зареждане на данните... (може да отнеме около минута, тъй като файлът е 23MB)
ДАННИТЕ СА ЗАРЕДЕНИ УСПЕШНО!


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom



Общ размер на масива: 541909 реда и 8 колони


In [ ]:
print("Липсващи стойности преди чистенето:")
print(df.isnull().sum())

#Премахваме редовете без CustomerID
df_clean = df.dropna(subset=["CustomerID"])

#Премахваме върнатите поръчки и безплатните артикули (оставяме само > 0)
df_clean = df_clean[(df_clean["Quantity"] > 0) & (df_clean["UnitPrice"] > 0)]

#Създаваме нова колона 'TotalSum' (Обща сума) = Количество * Единична цена
df_clean["TotalSum"] = df_clean["Quantity"] * df_clean["UnitPrice"]

#Превръщаме CustomerID в цяло число (махаме десетичната запетая) и после в текст, защото това е просто идентификатор, а не число за събиране
df_clean["CustomerID"] = df_clean["CustomerID"].astype(int).astype(str)

print(f"Останали чисти редове: {df_clean.shape[0]}")
display(df_clean[["InvoiceNo", "CustomerID", "Quantity", "UnitPrice", "TotalSum"]].head())

Липсващи стойности преди чистенето:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64
Останали чисти редове: 397884


,InvoiceNo,CustomerID,Quantity,UnitPrice,TotalSum
0,536365,17850,6,2.55,15.30
1,536365,17850,6,3.39,20.34
2,536365,17850,8,2.75,22.00
3,536365,17850,6,3.39,20.34
4,536365,17850,6,3.39,20.34


In [ ]:
import datetime as dt

#Определяме "днешната" дата, която се ползва за анализа
#(Взимаме датата на последната транзакция в целия масив + 1 ден)
latest_date = df_clean["InvoiceDate"].max() + dt.timedelta(days=1)

#Групираме данните по клиент (CustomerID) и изчисляваме RFM
rfm = df_clean.groupby("CustomerID").agg({
    "InvoiceDate": lambda x: (latest_date - x.max()).days,
    "InvoiceNo": "nunique",
    "TotalSum" : "sum"
})

#Преименуваме колоните, за да са ясни и професионални
rfm.rename(columns={
    "InvoiceDate": "Recency",
    "InvoiceNo": "Frequency",
    "TotalSum": "Monetary"
}, inplace=True)

#Премахваме дребни аномалии (като клиенти с обща сума 0)
rfm = rfm[rfm["Monetary"] > 0]

print("RFM таблицата е завършена")
print(f"Анализираме общо {rfm.shape[0]} уникални лоялни клиенти.")
display(rfm.head(10))

RFM таблицата е завършена
Анализираме общо 4338 уникални лоялни клиенти.


,Recency,Frequency,Monetary
CustomerID,,,
12346,326,1,77183.60
12347,2,7,4310.00
12348,75,4,1797.24
12349,19,1,1757.55
12350,310,1,334.40
12352,36,8,2506.04
12353,204,1,89.00
12354,232,1,1079.40
12355,214,1,459.40


In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import plotly.express as px

#ТРАНСФОРМАЦИЯ НА ДАННИТЕ
#Алгоритмите за машинно обучение работят най-добре, когато данните са в един мащаб.
#Тъй като парите (Monetary) могат да са 100 000, а честотата (Frequency) може да е 2,
#използваме логаритмуване и стандартизация, за да ги изравним.
rfm_log = np.log1p(rfm[['Recency', 'Frequency', 'Monetary']]) #Логаритмична трансформация (премахва изкривяванията) - Прилагаме само към числовите колони
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)

#ПРИЛАГАНЕ НА МАШИННО ОБУЧЕНИЕ (K-Means)
#Казваме на алгоритъма да раздели клиентите на точно 4 групи (клъстери)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
#Алгоритъмът поставя "етикет" (от 0 до 3) на всеки клиент, който впоследствие добавяме в таблицата като текст
rfm["Cluster"] = kmeans.fit_predict(rfm_scaled)
rfm["Cluster"] = rfm["Cluster"].astype(str)

print("БИЗНЕС ПРОФИЛ НА КЛИЕНТСКИТЕ ГРУПИ:")
cluster_summary = rfm.groupby("Cluster").agg({
    "Recency": "mean",
    "Frequency": "mean",
    "Monetary": ["mean", "count"]
}).round(1)
display(cluster_summary)

#ИНТЕРАКТИВНА 3D ВИЗУАЛИЗАЦИЯ
fig = px.scatter_3d(
    rfm,
    x="Recency",
    y="Frequency",
    z="Monetary",
    color="Cluster",
    opacity=0.8,
    title="3D Customer Segmentation (Machine Learning Clusters)",
    color_discrete_sequence=px.colors.qualitative.Pastel
)

fig.update_layout(margin=dict(l=0, r=0, b=0, t=40))
fig.show()

БИЗНЕС ПРОФИЛ НА КЛИЕНТСКИТЕ ГРУПИ:


Recency Frequency Monetary      
           mean      mean     mean count
Cluster                                 
0          18.1       2.1    551.8   837
1          12.1      13.7   8074.3   716
2          71.1       4.1   1802.8  1173
3         182.5       1.3    343.5  1612